In [18]:
import pandas as pd
import plotly.express as px

In [19]:
from google.colab import drive
drive.mount('/content/drive')

ruta = "/content/drive/MyDrive/Colab Notebooks/IA/Mburicao/"

df0 = pd.read_csv(ruta + "Mburicao_Sil.csv",
                  header=None,
                  names=["fecha","nivel"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
inicio = '2025-12-31 23:00:00' ##se adelanta 1 hora luego
fin = '2026-01-31 22:55:00'
mascara = (df0['fecha'] >= inicio) & (df0['fecha'] <= fin)
df_enero = df0.loc[mascara].copy()

In [21]:
df_enero['fecha'] = pd.to_datetime(df_enero['fecha'])
df_enero= df_enero.sort_values('fecha').reset_index(drop=True)

In [22]:
def round_up_to_next_minute(ts):
    if ts.second != 0 or ts.microsecond != 0:
        # Suma los segundos restantes para llegar al minuto siguiente
        return ts + pd.Timedelta(seconds=(60 - ts.second), microseconds=-ts.microsecond)
    return ts

df_enero['fecha'] = df_enero['fecha'].apply(round_up_to_next_minute)

In [23]:
# Contar ocurrencias de cada fecha
conteo = df_enero['fecha'].value_counts()

# Filtrar aquellas que aparecen más de una vez
repetidas = conteo[conteo > 1]

print(f"Total de fechas con repeticiones: {len(repetidas)}")
print("\nFechas repetidas y su frecuencia:")
print(repetidas)

# Opcional: mostrar las filas correspondientes a esas fechas para inspeccionar
if not repetidas.empty:
    # Crear una lista de las fechas repetidas
    fechas_repetidas = repetidas.index.tolist()
    # Filtrar el DataFrame original por esas fechas y ordenar
    df_repetidas = df_enero[df_enero['fecha'].isin(fechas_repetidas)].sort_values('fecha')
    display(df_repetidas)

Total de fechas con repeticiones: 70

Fechas repetidas y su frecuencia:
fecha
2026-01-31 00:15:00    2
2026-01-28 12:00:00    2
2026-01-05 03:45:00    2
2026-01-24 09:25:00    2
2026-01-22 15:30:00    2
                      ..
2026-01-28 10:00:00    2
2026-01-23 01:50:00    2
2026-01-17 14:30:00    2
2026-01-28 13:05:00    2
2026-01-20 17:30:00    2
Name: count, Length: 70, dtype: int64


,fecha,nivel
61,2026-01-01 04:05:00,6.505
62,2026-01-01 04:05:00,6.505
335,2026-01-02 03:00:00,6.468
336,2026-01-02 03:00:00,6.468
605,2026-01-03 01:50:00,6.500
...,...,...
8238,2026-01-30 03:15:00,6.522
8486,2026-01-31 00:15:00,6.517
8487,2026-01-31 00:15:00,6.517
8497,2026-01-31 01:05:00,6.519


In [24]:
# =============================================
# ELIMINAR FILAS CON FECHA DUPLICADA (conservar la primera)
# =============================================

# Verificar cuántas filas tenemos antes
print(f"Filas antes de eliminar duplicados: {len(df_enero)}")

# Identificar fechas repetidas (opcional, solo para diagnóstico)
duplicados_fecha = df_enero.duplicated(subset=['fecha'], keep=False)
print(f"Filas con fecha duplicada (incluyendo primera aparición): {duplicados_fecha.sum()}")
if duplicados_fecha.sum() > 0:
    print("Ejemplo de fechas repetidas (primeras 5):")
    display(df_enero[duplicados_fecha].sort_values('fecha').head(10))

# Eliminar duplicados basados en la columna 'fecha', conservando la primera ocurrencia
df_enero = df_enero.drop_duplicates(subset=['fecha'], keep='first')

# Verificar después
print(f"Filas después de eliminar duplicados: {len(df_enero)}")

Filas antes de eliminar duplicados: 8758
Filas con fecha duplicada (incluyendo primera aparición): 140
Ejemplo de fechas repetidas (primeras 5):


,fecha,nivel
61,2026-01-01 04:05:00,6.505
62,2026-01-01 04:05:00,6.505
335,2026-01-02 03:00:00,6.468
336,2026-01-02 03:00:00,6.468
605,2026-01-03 01:50:00,6.500
606,2026-01-03 01:50:00,6.500
908,2026-01-04 03:10:00,6.503
909,2026-01-04 03:10:00,6.503
1047,2026-01-04 14:55:00,6.493
1048,2026-01-04 14:55:00,6.493


Filas después de eliminar duplicados: 8688


In [25]:
# Contar cuántos duplicados quedan (debe ser 0)
print("Número de filas con fecha duplicada:", df_enero.duplicated(subset=['fecha']).sum())

Número de filas con fecha duplicada: 0


In [26]:
df_enero['fecha'] = df_enero['fecha'] + pd.Timedelta(hours=1) #se adelanta la hora

In [27]:
df_enero

,fecha,nivel
0,2026-01-01 00:00:00,6.490
1,2026-01-01 00:05:00,6.491
2,2026-01-01 00:10:00,6.492
3,2026-01-01 00:15:00,6.492
4,2026-01-01 00:20:00,6.493
...,...,...
8753,2026-01-31 23:35:00,6.519
8754,2026-01-31 23:40:00,6.519
8755,2026-01-31 23:45:00,6.518
8756,2026-01-31 23:50:00,6.518


In [28]:
# =============================================
# Crear índice regular de 5 minutos y reindexar
# =============================================

# Asegurar que la columna 'fecha' sea el índice
df_enero = df_enero.set_index('fecha').sort_index()

# Definir rango completo
start_time = df_enero.index.min()
end_time = df_enero.index.max()
time_index = pd.date_range(start=start_time, end=end_time, freq='5min')

# Reindexar: las marcas faltantes quedarán con NaN
df_regular = df_enero.reindex(time_index)
df_regular.index.name = 'fecha'

# Verificar cuántos NaN se han introducido
print(f"Filas después de reindexar: {len(df_regular)}")
print(f"Valores nulos en 'nivel': {df_regular['nivel'].isna().sum()}")

Filas después de reindexar: 8928
Valores nulos en 'nivel': 240


In [29]:
# Interpolar linealmente los valores nulos en la columna 'nivel'
df_regular['nivel'] = df_regular['nivel'].interpolate(method='linear')

# Verificar que ya no hay nulos
print("Valores nulos después de interpolar:", df_regular['nivel'].isna().sum())

Valores nulos después de interpolar: 0


In [30]:
# =============================================
#  APLICAR TRANSFORMACIÓN (7 - nivel)
# =============================================
df_regular['nivel'] = 7 - df_regular['nivel']

In [31]:
# =============================================
# VERIFICAR FRECUENCIA UNIFORME (5 minutos)
# =============================================
diffs = df_regular.index.to_series().diff()
print("\nFrecuencia de las diferencias entre timestamps (debe ser 5 minutos):")
print(diffs.value_counts())


Frecuencia de las diferencias entre timestamps (debe ser 5 minutos):
fecha
0 days 00:05:00    8927
Name: count, dtype: int64


In [32]:
# =============================================
# GRAFICAR INTERACTIVAMENTE CON PLOTLY
# =============================================
import plotly.express as px

fig = px.line(df_regular, x=df_regular.index, y='nivel',
              title='Nivel de agua (m) – Enero 2026 (corregido +1h, interpolado)')
fig.update_layout(hovermode='x unified')
fig.show()

In [ ]:
import os

# Definir ruta en Drive
ruta_drive = '/content/drive/MyDrive/Colab Notebooks/IA/Mburicao/Datos_Enero_2026'
os.makedirs(ruta_drive, exist_ok=True)

# 1. DataFrame de 5 minutos (ya debe existir df_regular)
df_regular['nivel'] = df_regular['nivel'].round(4)  # Asegura 4 decimales

# Guardar Excel 5min
archivo_5min_excel = os.path.join(ruta_drive, 'Mburicao_enero_2026_5min.xlsx')
df_regular.to_excel(archivo_5min_excel)
print(f"Archivo de 5 min guardado en: {archivo_5min_excel}")

# Guardar CSV 5min
archivo_5min_csv = os.path.join(ruta_drive, 'Mburicao_enero_2026_5min.csv')
df_regular.to_csv(archivo_5min_csv, index=True)  # index=True incluye la fecha como primera columna
print(f"Archivo CSV de 5 min guardado en: {archivo_5min_csv}")